# Caching Strategies: KV Cache, Semantic Caching, and Cache Poisoning

Caching is the highest-leverage cost optimization in LLM infrastructure. The same prompt prefix (system prompt + retrieval context) is often sent thousands of times per day. Caching eliminates the redundant compute.

## Prompt Cache Mechanics (Anthropic)

Anthropic's prompt caching saves and reuses the KV cache for repeated context prefixes:
- Mark a block with `cache_control: {type: 'ephemeral'}` to cache it
- Cached blocks cost ~10% of the normal input token price on cache hit
- Cache lifetime: 5 minutes, extended on each hit
- Minimum cacheable block: 1024 tokens

The key pattern: place stable, expensive content (system prompt, large documents, tool definitions) at the beginning of the context — this is what gets cached. User-specific, turn-specific content comes after the cached prefix.

In [ ]:
from dataclasses import dataclass, field
from typing import Callable
import hashlib, time, math

@dataclass
class CacheEntry:
    key: str
    value: str
    created_at: float
    accessed_at: float
    hit_count: int = 0
    token_count: int = 0

class SemanticCache:
    def __init__(self, embed_fn: Callable, similarity_threshold: float = 0.92, ttl: int = 300):
        self.embed = embed_fn
        self.threshold = similarity_threshold
        self.ttl = ttl
        self.cache: list = []
        self.stats = {'hits': 0, 'misses': 0, 'evictions': 0}

    def _cosine_sim(self, a: list, b: list) -> float:
        dot = sum(x*y for x, y in zip(a, b))
        na = math.sqrt(sum(x*x for x in a))
        nb_ = math.sqrt(sum(x*x for x in b))
        return dot / (na * nb_ + 1e-8)

    def _evict_expired(self):
        now = time.time()
        before = len(self.cache)
        self.cache = [(emb, e) for emb, e in self.cache if now - e.created_at < self.ttl]
        self.stats['evictions'] += before - len(self.cache)

    def get(self, query: str) -> tuple:
        self._evict_expired()
        q_emb = self.embed(query)
        best_sim, best_entry = 0.0, None
        for emb, entry in self.cache:
            sim = self._cosine_sim(q_emb, emb)
            if sim > best_sim:
                best_sim, best_entry = sim, entry
        if best_sim >= self.threshold and best_entry:
            best_entry.hit_count += 1
            best_entry.accessed_at = time.time()
            self.stats['hits'] += 1
            return best_entry.value, best_sim
        self.stats['misses'] += 1
        return None, best_sim

    def set(self, query: str, value: str, token_count: int = 0):
        emb = self.embed(query)
        key = hashlib.md5(query.encode()).hexdigest()[:8]
        entry = CacheEntry(key=key, value=value, created_at=time.time(),
                            accessed_at=time.time(), token_count=token_count)
        self.cache.append((emb, entry))

    def hit_rate(self) -> float:
        total = self.stats['hits'] + self.stats['misses']
        return self.stats['hits'] / total if total > 0 else 0.0

    def cost_savings(self, price_per_token: float = 0.000003) -> float:
        saved_tokens = sum(e.hit_count * e.token_count for _, e in self.cache)
        return saved_tokens * price_per_token

rng = random.Random(42)
def mock_embed(text: str) -> list:
    seed = hash(text[:30]) % 100000
    r = random.Random(seed)
    return [r.gauss(0, 1) for _ in range(16)]

cache = SemanticCache(mock_embed, similarity_threshold=0.85)

queries = [
    ('What is gradient descent?', 'Gradient descent is an optimization algorithm...', 150),
    ('Explain gradient descent', None, 0),  # should hit cache
    ('What is backpropagation?', 'Backpropagation computes gradients via chain rule...', 120),
    ('How does gradient descent work?', None, 0),  # should hit cache
    ('What is the capital of France?', 'Paris is the capital of France.', 20),
]

for query, response, tokens in queries:
    hit, sim = cache.get(query)
    if hit:
        print(f'CACHE HIT (sim={sim:.3f}): {query[:50]}')
    else:
        print(f'CACHE MISS (sim={sim:.3f}): {query[:50]}')
        if response:
            cache.set(query, response, tokens)

print(f'\nCache stats: hit_rate={cache.hit_rate():.0%}, entries={len(cache.cache)}')
print(f'Estimated savings: ${cache.cost_savings():.6f}')

## Cache Poisoning Risks

Semantic caches have a security vulnerability: if an attacker can inject a malicious response into the cache for a query that others will ask, subsequent users get the malicious response.

Mitigation:
1. **Cache keys include user/session context** for sensitive queries
2. **Response validation** before caching: check outputs against safety classifiers
3. **Cache TTL**: short TTL (5-15 min) limits the exposure window
4. **Cache isolation**: user-specific queries go to user-isolated caches, not a shared cache

## Prompt Caching Architecture

For production deployments with Anthropic's API:
1. Place system prompt and tool definitions in the first `cache_control` block
2. If using RAG, place the retrieved documents in a second cacheable block (if the document set is stable per session)
3. User turn is never cached (changes each request)
4. Measure cache hit rate via `usage.cache_read_input_tokens` in API responses
5. Target >80% cache hit rate for stable system prompts and tool definitions

At 1M requests/day with a 40K token system prompt, 80% cache hit rate saves approximately $96K/month vs uncached.